# Notebook demonstrating tidal composites generation

Apply geomedian to Sentinel-2 imagery at the highest and lower tide ranges.

In [ ]:
import datacube
from datacube.utils.dask import start_local_dask
import xarray as xr
from coastlines.composites import get_study_site_geometry, GeoBox, load_s2, tidal_composites, rename_add_prefix

In [ ]:
# Create local dask client for parallelisation
dask_client = start_local_dask(
    n_workers=4, threads_per_worker=8, mem_safety_margin="2GB"
)
dask_client

## Set processing parameters and study area

In [ ]:
# use 3 years to ensure sufficient data at high and low tides
start_date = "2023-01-01"
end_date = "2025-12-31"

output_crs = "epsg:6933"
resolution = 10
include_coastal_aerosol=True
max_cloudcover = 60
dtype = "int16"
cloud_filters = {"cloud medium probability": [["opening", 2], ["dilation", 5]],
                     "cloud high probability": [["opening", 2], ["dilation", 5]],
                     "thin cirrus": [["dilation", 5]]}

# Tide data and config
tide_model_location = "/home/jovyan/data/coastlines/tide_models"
tide_model = "FES2022"

In [ ]:
study_area = "1956,62"
grid_path = "https://raw.githubusercontent.com/piksel-ina/Indonesia-coastlines/main/data/raw/indonesia_10km_tiles_coast.geojson"

In [ ]:
geometry = get_study_site_geometry(grid_path, study_area)
geobox = GeoBox.from_bbox(geometry.to_crs(output_crs).bounds.values[0], crs=output_crs, resolution=resolution)
# View the AoI
geobox.explore()

## Load Sentinel-2 data

In [ ]:
# Connect to ODC
dc = datacube.Datacube(app="tidal_composites")

In [ ]:
# Load satellite data
satellite_ds, dss_s2 = load_s2(
    dc=dc,
    geobox=geobox,
    time_range=(start_date, end_date),
    resolution=resolution,
    crs=output_crs,
    include_coastal_aerosol=include_coastal_aerosol,
    max_cloudcover=max_cloudcover,
    cloud_filters=cloud_filters,
    dtype=dtype,
)

In [ ]:
satellite_ds

In [ ]:
ds_lowtide, ds_hightide = tidal_composites(
                satellite_ds=satellite_ds,
                threshold_lowtide=0.15,
                threshold_hightide=0.85,
                min_obs=0,
                tide_model=tide_model,
                tide_model_dir=tide_model_location,
            )

In [ ]:
# Rename low and high tide bands to add "low"/"high"
ds_hightide = rename_add_prefix(ds_hightide, "high")
ds_lowtide = rename_add_prefix(ds_lowtide, "low")

In [ ]:
# Concatenate into a single output dataset
ds_tidalcomposites = xr.merge([ds_lowtide, ds_hightide], compat='no_conflicts')

In [ ]:
import folium

In [ ]:
# Explore the results
lat, lon = geobox.geographic_extent.centroid.xy[1][0], geobox.geographic_extent.centroid.xy[0][0]
m = folium.Map(location=(lat, lon), zoom_start=13)
ds_lowtide.odc.to_rgba(bands=["low_red", "low_green", "low_blue"], vmin=0, vmax=3000).odc.add_to(
    m, name="Low Tide Composite"
)
ds_hightide.odc.to_rgba(bands=["high_red", "high_green", "high_blue"], vmin=0, vmax=3000).odc.add_to(
    m, name="High Tide Composite"
)
folium.LayerControl().add_to(m)
m

In [ ]:
ds_tidalcomposites